# Gated cross-attention in NumPy

A Flamingo-style model keeps the language model's sequence text only. Each photo leaves a
marker token in the text, a resampler turns the photo's patch vectors into a fixed number
of vectors, and new cross-attention layers between the frozen layers let the text read
them through a gate that starts closed. You will write the three pieces on a small model:
the resampler, the mask that says which photo each text position reads, and the gated
layer.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
width, n_latents = 8, 4   # a real model: a width in the thousands and 64 latents
PHOTO = 5                 # the marker id a photo leaves in the text


def matrix():
    return rng.standard_normal((width, width)) / np.sqrt(width)


R_q, R_k, R_v = matrix(), matrix(), matrix()       # the resampler's weights
latents = rng.standard_normal((n_latents, width))  # the resampler's learned latents
X_q, X_k, X_v, X_o = matrix(), matrix(), matrix(), matrix()   # the new layer's weights

# The image encoder's output for two photos: 12 and 30 patch vectors.
features = [rng.standard_normal((12, width)), rng.standard_normal((30, width))]
# A text with two photo markers, and the frozen layer's hidden states at its positions.
tokens = np.array([17, 18, PHOTO, 19, 20, 21, PHOTO, 22, 23])
x = rng.standard_normal((len(tokens), width))
print("patch vectors per photo:", [len(f) for f in features], " text positions:", len(tokens))

## Part 1 (coding task): The resampler

Write `resample(feats, latents, Wq, Wk, Wv)`, one attention layer whose queries come from
the learned latents rather than from the photo:

- The queries are `latents @ Wq`, one per latent.
- The keys and values are `kv @ Wk` and `kv @ Wv`, where `kv` stacks the photo's rows
  `feats` and then the rows of `latents`.
- Each query's softmax runs over all the keys, on scores divided by $\sqrt{\text{width}}$.
- The function returns `latents` plus the attention output.

The output has one row per latent, however many patch vectors the photo has.

In [ ]:
def resample(feats, latents, Wq, Wk, Wv):
    """n_latents vectors that summarize one photo's patch vectors."""
    # TODO
    raise NotImplementedError

In [ ]:
r0 = np.asarray(resample(features[0], latents, R_q, R_k, R_v))
r1 = np.asarray(resample(features[1], latents, R_q, R_k, R_v))
assert r0.shape == (n_latents, width) and r1.shape == (n_latents, width), f"shapes {r0.shape} and {r1.shape}: expected one row per latent for both photos"


def loop_resample(f):
    kv = np.vstack([f, latents])
    out = []
    for q in latents @ R_q:
        s = (kv @ R_k) @ q / np.sqrt(width)
        w = np.exp(s - s.max())
        out.append((w / w.sum()) @ (kv @ R_v))
    return latents + np.array(out)


assert np.allclose(r0, loop_resample(features[0])), "wrong output for photo 0: check the softmax axis, the scale, which rows are keys, and the added latents"
assert np.allclose(r1, loop_resample(features[1])), "wrong output for photo 1"
order = np.random.default_rng(0).permutation(len(features[1]))
assert np.allclose(resample(features[1][order], latents, R_q, R_k, R_v), r1), "shuffling the patch vectors should not change the output"
print("Part 1 passed")

## Part 2 (coding task): Which photo each position reads

Text position $t$ reads the photo of the last marker at or before $t$: a marker's own
position reads its photo, and so does every position up to the next marker. A position
before the first marker reads no photo.

Write `photo_mask(tokens, marker, n_latents)`. The resampled photos will be stacked in
order, `n_latents` rows each. Return a boolean array with one row per text position and one
column per stacked row, `True` where that position may read that row.

In [ ]:
def photo_mask(tokens, marker, n_latents):
    """mask[t, j] is True when text position t may read stacked photo row j."""
    # TODO
    raise NotImplementedError

In [ ]:
m = np.asarray(photo_mask(tokens, PHOTO, n_latents))
assert m.shape == (len(tokens), 2 * n_latents), f"mask shape {m.shape}: one row per text position, one column per stacked photo row"
assert m.dtype == bool, f"mask dtype {m.dtype}: return a boolean array"
assert not m[:2].any(), "positions 0 and 1 come before the first marker and should read nothing"
assert m[2, :n_latents].all() and not m[2, n_latents:].any(), "position 2 is the first marker and should read photo 0"
assert m[3:6, :n_latents].all() and not m[3:6, n_latents:].any(), "positions 3 to 5 should read photo 0 only"
assert m[6:, n_latents:].all() and not m[6:, :n_latents].any(), "positions 6 to 8 should read photo 1 only"
pair = np.asarray(photo_mask(np.array([PHOTO, PHOTO, 9]), PHOTO, 3))
expected = np.array([[1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 1], [0, 0, 0, 1, 1, 1]], dtype=bool)
assert np.array_equal(pair, expected), f"two markers in a row: got {pair.astype(int).tolist()}"
print(m.astype(int))
print("Part 2 passed")

## Part 3 (coding task): The gated layer

Write `gated_cross_attention(x, photo_rows, mask, Wq, Wk, Wv, Wo, alpha)`. `x` holds the
text's hidden states, one row per position, and `photo_rows` the resampled photos stacked
in order.

- The queries are `x @ Wq`, the keys and values `photo_rows @ Wk` and `photo_rows @ Wv`,
  and the scores are divided by $\sqrt{\text{width}}$.
- Each position's softmax runs over the photo rows its mask row allows. A position whose
  mask row allows none reads nothing.
- Return `x + np.tanh(alpha) * (out @ Wo)`, where row $t$ of `out` is what position $t$
  read.

The check runs the whole layer: both photos through your resampler, your mask, then the
layer.

In [ ]:
def gated_cross_attention(x, photo_rows, mask, Wq, Wk, Wv, Wo, alpha):
    """The text's hidden states after the gated cross-attention layer."""
    # TODO
    raise NotImplementedError

In [ ]:
photo_rows = np.vstack([resample(f, latents, R_q, R_k, R_v) for f in features])
mask = np.asarray(photo_mask(tokens, PHOTO, n_latents))


def loop_layer(rows, alpha):
    out = np.zeros_like(x)
    for t in range(len(x)):
        if mask[t].any():
            keep = rows[mask[t]]
            s = (keep @ X_k) @ (x[t] @ X_q) / np.sqrt(width)
            w = np.exp(s - s.max())
            out[t] = (w / w.sum()) @ (keep @ X_v)
    return x + np.tanh(alpha) * (out @ X_o)


y0 = np.asarray(gated_cross_attention(x, photo_rows, mask, X_q, X_k, X_v, X_o, 0.0))
bad = sorted(set(np.argwhere(~np.isfinite(y0))[:, 0].tolist()))
assert not bad, f"positions {bad} came out NaN or infinite"
assert np.array_equal(y0, x), "with alpha = 0 the layer should return x exactly"
y = np.asarray(gated_cross_attention(x, photo_rows, mask, X_q, X_k, X_v, X_o, 0.8))
assert np.allclose(y[:2], x[:2]), "positions 0 and 1 come before any photo, so the layer should leave them as they were"
assert np.allclose(y, loop_layer(photo_rows, 0.8)), "wrong output at alpha = 0.8"
moved = photo_rows.copy()
moved[n_latents:] += 5.0
y_moved = np.asarray(gated_cross_attention(x, moved, mask, X_q, X_k, X_v, X_o, 0.8))
assert np.allclose(y_moved[:6], y[:6]), "changing photo 1 changed a position before its marker"
print("change per position at alpha = 0.8:", np.round(np.linalg.norm(y - x, axis=1), 3))
print("Part 3 passed")

## Solution

### Part 1: The resampler

In [ ]:
def resample(feats, latents, Wq, Wk, Wv):
    """n_latents vectors that summarize one photo's patch vectors."""
    kv = np.vstack([feats, latents])
    fit = (latents @ Wq) @ (kv @ Wk).T / np.sqrt(latents.shape[1])
    share = np.exp(fit - fit.max(axis=1, keepdims=True))
    share = share / share.sum(axis=1, keepdims=True)   # each latent's row of weights adds up to 1
    return latents + share @ (kv @ Wv)

**Why it works.** The queries come from the latents, so there are always `n_latents` of
them, and each output row is a softmax average over however many keys there are. The
number of patch vectors changes how long each average is, never how many rows come out,
which is what keeps the cross-attention's cost fixed as photos get larger. An average is a
sum, so reordering the patch rows only reorders its terms: the resampler sees no patch
order unless the features carry it themselves. Putting the latents among the keys is a
choice this notebook makes, so each latent can also read the others. Flamingo's paper
describes the latents only as the queries that cross-attend to the visual features.

### Part 2: Which photo each position reads

In [ ]:
def photo_mask(tokens, marker, n_latents):
    """mask[t, j] is True when text position t may read stacked photo row j."""
    is_marker = np.asarray(tokens) == marker
    reads = np.cumsum(is_marker) - 1                   # photo index per position, -1 for none
    photo_of_row = np.arange(is_marker.sum() * n_latents) // n_latents
    return reads[:, None] == photo_of_row[None, :]

**Why it works.** Counting the markers up to and including each position gives the index
of the photo it reads, and the count minus one is $-1$ before the first marker. Each stacked
row belongs to photo `j // n_latents`, so comparing the two indices with broadcasting builds
the whole mask at once. Positions before any marker match no column. With this rule the text
after two markers in a row reads only the second photo, so the first one reaches the text
only through its own marker's position.

### Part 3: The gated layer

In [ ]:
def gated_cross_attention(x, photo_rows, mask, Wq, Wk, Wv, Wo, alpha):
    """The text's hidden states after the gated cross-attention layer."""
    mask = np.asarray(mask, dtype=bool)
    scores = (x @ Wq) @ (photo_rows @ Wk).T / np.sqrt(x.shape[1])
    reads = mask.any(axis=1)                            # positions with a photo to read
    out = np.zeros_like(x)
    s = np.where(mask[reads], scores[reads], -np.inf)
    w = np.exp(s - s.max(axis=1, keepdims=True))        # hidden scores give exp(-inf) = 0
    out[reads] = (w / w.sum(axis=1, keepdims=True)) @ (photo_rows @ Wv)
    return x + np.tanh(alpha) * (out @ Wo)

**Why it works.** A softmax over no keys is undefined. With every score of a row set to
$-\infty$ it computes $0/0$ and gives NaN, and with a large negative number in place of
$-\infty$ it spreads equal weights over every photo row, so the text before the first marker
would read both photos, including one that comes later. Computing the softmax only for the
rows that allow something, and leaving the other rows of `out` at zero, gives those positions
nothing to add. With `alpha = 0` the factor $\tanh 0 = 0$ removes the layer's contribution,
so the output is `x` to the last bit: the model starts as its language model. That holds only
when the contribution is finite, since $0 \times \text{NaN}$ is still NaN, which is why a NaN
at the first two positions shows up even with the gate closed.